<a href="https://colab.research.google.com/github/EthanIsaac/valorant-voiceline-qa/blob/main/notebooks/01_agent_voice_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Agent Voice Classifier: fine-tuning 3 audio models

**Project:** VALORANT Voice-Line QA & Localization Assistant (Riot Games)
**Pipeline being fine-tuned:** `audio-classification`, which identifies **which of the 26 agents** is speaking in a voice-line recording.

**Why it matters:** each agent ships with hundreds of voice lines, recorded and localized into many languages. Audio QA must confirm that every file contains the right agent's voice before it reaches the game.

**Notebook outline**
- Step 0: Setup (packages, seed, output folder)
- Step 1: Load the dataset at a pinned revision
- Step 2: Clean the data and split it into train / validation / test
- Step 3: Choose the 3 candidate pre-trained models and prepare the audio
- Step 4: Fine-tune and evaluate each model (accuracy and macro-F1)
- Step 5: Measure CPU runtime through a `pipeline` (Streamlit Cloud runs on CPU)
- Step 6: Compare the models and pick the best
- Step 7: Upload the best model and its model card to Hugging Face
- Save and download the results (with SHA-256 fingerprints)

All results are written to `results/agent_classifier/` so the report can quote them exactly.

> Runtime: **GPU** (Runtime → Change runtime type → T4 GPU).

# Step 0: Setup

In [1]:
# Same transformers constraint as the course notebooks
# (pandas, scikit-learn, soundfile, librosa and huggingface_hub are already installed in Colab).
# Expected: a pip "dependency conflicts" message about diffusers/gradio needing a newer
# huggingface-hub. Those packages are not used here, so it can be ignored.
%pip install -q "transformers>=4.46.0,<5.0.0" accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 59.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import io
import json
import os
import re
import time

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from transformers import set_seed

# One seed for everything (Python, NumPy, PyTorch), so reruns give the same results
SEED = 42
set_seed(SEED)

# Every result file produced by this notebook goes in this folder
RESULTS_DIR = "results/agent_classifier"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Hugging Face token (only needed to upload the model in Step 7), read from the Colab secret HF_TOKEN.
# Read explicitly here: if the secret is granted after the first Hub download, the library would not see it.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Hugging Face token: found")
except Exception:
    print("Hugging Face token: not found (the upload in Step 7 will be skipped)")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

transformers 4.57.6 | torch 2.11.0+cu130 | device: cuda
GPU: Tesla T4


# Step 1: Load the dataset

**Source:** [NgThVinh/ValorantAgentVoiceLines](https://huggingface.co/datasets/NgThVinh/ValorantAgentVoiceLines), in-game voice lines of every VALORANT agent with their transcripts (collected from the VALORANT Fandom wiki).
Each agent is stored as a separate file. The **agent name is the label** we want to predict.

We load a **fixed revision** (commit hash) so the data can never change under us. The download is about 2 GB and takes a couple of minutes.

In [3]:
from huggingface_hub import hf_hub_download, list_repo_files

DATASET = "NgThVinh/ValorantAgentVoiceLines"
REVISION = "c2e0d3493f44da5be07090d6ebb9b69ebbcb2060"   # pinned commit

# One parquet file per agent: "<agent>/train-00000-of-00001.parquet"
parquet_files = sorted(
    f for f in list_repo_files(DATASET, repo_type="dataset", revision=REVISION) if f.endswith(".parquet")
)

frames = []
for file in parquet_files:
    path = hf_hub_download(DATASET, file, repo_type="dataset", revision=REVISION)
    agent_df = pd.read_parquet(path)
    agent_df["agent"] = file.split("/")[0]
    frames.append(agent_df)

data = pd.concat(frames, ignore_index=True)
data["audio_bytes"] = data["audio_file"].map(lambda audio: audio["bytes"])   # raw WAV file content
data = data.drop(columns="audio_file")

AGENTS = sorted(data["agent"].unique())
print(f"{len(data)} voice lines from {len(AGENTS)} agents")
print(data.groupby("agent").size().to_string())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


astra/train-00000-of-00001.parquet:   0%|          | 0.00/83.9M [00:00<?, ?B/s]

breach/train-00000-of-00001.parquet:   0%|          | 0.00/65.3M [00:00<?, ?B/s]

brimstone/train-00000-of-00001.parquet:   0%|          | 0.00/58.7M [00:00<?, ?B/s]

chamber/train-00000-of-00001.parquet:   0%|          | 0.00/69.8M [00:00<?, ?B/s]

...

tejo/train-00000-of-00001.parquet:   0%|          | 0.00/51.4M [00:00<?, ?B/s]

viper/train-00000-of-00001.parquet:   0%|          | 0.00/73.9M [00:00<?, ?B/s]

vyse/train-00000-of-00001.parquet:   0%|          | 0.00/69.7M [00:00<?, ?B/s]

yoru/train-00000-of-00001.parquet:   0%|          | 0.00/62.2M [00:00<?, ?B/s]

12164 voice lines from 26 agents
agent
astra        502
breach       484
brimstone    461
chamber      447
clove        461
cypher       480
deadlock     436
fade         440
gekko        563
harbor       432
iso          423
jett         472
kayo         469
killjoy      488
neon         458
omen         496
phoenix      460
raze         493
reyna        481
sage         474
skye         463
sova         479
tejo         389
viper        484
vyse         457
yoru         472


In [4]:
# Listen to a few examples
for _, row in data.sample(3, random_state=SEED).iterrows():
    print(f"{row['agent']:<10} {row['audio_name']:<28} {row['transcript']}")
    display(Audio(row["audio_bytes"]))

reyna      ReynaBeQuiet2                "Be quiet."


gekko      GekkoNeedUtilityHere2        "Need util here."


raze       RazePainLight2               ''Raze groans''


# Step 2: Clean the data and split it

**Audio checks.** Every clip is a mono WAV recorded at 48 kHz. Clips shorter than **0.5 s** (e.g. "Mm-hmm") carry too little voice to identify a speaker, so they are removed.

**Transcript cleaning.** The transcripts contain wiki markup. The words actually spoken are always the parts **inside double quotes**:

| Raw transcript | Spoken text |
|---|---|
| `"Hey hermanos, spend everything you have." (''Hermanos'': Siblings)` | `Hey hermanos, spend everything you have.` (translator note removed) |
| `''sighs'' "Cool"` | `Cool` (stage direction removed) |
| `"Sure, but if anything explodes again, you are telling Brimstone\|Brim."` | `... you are telling Brim.` (wiki link: the second name is the one spoken) |
| `''Jett groans''` | *(no speech)* |

Two flags are added for the speech-to-text notebook: `has_speech` (the clip contains words) and `english_only` (no foreign words or translator notes).
Non-speech clips (groans, sighs) are **kept** for agent identification, because they are still the agent's voice.

**Train / validation / test split (80 / 10 / 10).** Many lines were recorded several times (e.g. two takes of *"Keep up."* by the same agent).
If one take were in training and another in test, the test would be too easy (**leakage**). So the split is made **per agent and by line**:
all takes of the same line by the same agent go to the same split. Every agent appears in all three splits.

In [5]:
MIN_DURATION_S = 0.5
SPLIT_FRACTIONS = {"train": 0.8, "validation": 0.1, "test": 0.1}


def spoken_text(transcript):
    """Keep only the quoted (spoken) parts; in wiki links 'Name|Nickname' keep the spoken nickname."""
    quoted = re.findall(r'"([^"]*)"', str(transcript))
    text = " ".join(part.strip() for part in quoted)
    text = re.sub(r"[^\s|]+\|([^\s,.!?]+)", r"\1", text)
    return " ".join(text.split())


data["duration_s"] = data["audio_bytes"].map(lambda b: sf.info(io.BytesIO(b)).duration)
data["spoken_text"] = data["transcript"].map(spoken_text)
data["has_speech"] = data["spoken_text"] != ""
data["english_only"] = (
    data["has_speech"]
    & ~data["transcript"].str.contains(r"\(", regex=True)                       # no translator note
    & ~data["spoken_text"].str.contains(r"[^\x00-\x7F’‘“”…—–]", regex=True)   # no non-English characters
)

cleaning_log = [("original", len(data))]
data = data[data["duration_s"] >= MIN_DURATION_S].reset_index(drop=True)
cleaning_log.append((f"removed clips shorter than {MIN_DURATION_S}s", len(data)))

print(pd.DataFrame(cleaning_log, columns=["step", "clips"]).to_string(index=False))
print(f"\nDuration (s): median {data['duration_s'].median():.2f}, 95th pct {data['duration_s'].quantile(0.95):.2f}, max {data['duration_s'].max():.2f}")
print(f"Clips with speech: {data['has_speech'].sum()} | English-only: {data['english_only'].sum()}")

# Before/after examples of the transcript cleaning
data.loc[data["transcript"].str.contains(r"\(|''|\|", regex=True), ["transcript", "spoken_text"]].sample(8, random_state=SEED)

                           step  clips
                       original  12164
removed clips shorter than 0.5s  11360

Duration (s): median 1.02, 95th pct 4.80, max 16.02
Clips with speech: 11049 | English-only: 10740


,transcript,spoken_text
10759,''Vyse groans'',
4253,"""अरे, look at that rock-cut architecture! This...","अरे, look at that rock-cut architecture! This ..."
804,''Breach sighs'',
4568,"""平庸。"" (''Mediocre.)",平庸。
1535,"''laughs'' ""I'm sorry, but I could not help my...","I'm sorry, but I could not help myself"
5498,''KAY/O groans'',
4031,"''laughs'' ""We'll forge our friendship in the ...",We'll forge our friendship in the fires of Mis...
4832,"""폼 미쳤는데? Neon! Leave some for me, yeah?"" (''폼...","폼 미쳤는데? Neon! Leave some for me, yeah?"


In [6]:
# Group = one line of one agent (all takes of that line). Non-speech clips are grouped by their raw description.
line = np.where(data["has_speech"], data["spoken_text"], data["transcript"]).astype(str)
data["line_group"] = data["agent"] + "::" + pd.Series(line).str.lower().str.strip()

rng = np.random.default_rng(SEED)
group_split = {}
for agent, groups in data.groupby("agent")["line_group"]:
    groups = sorted(groups.unique())
    rng.shuffle(groups)
    n_test = round(len(groups) * SPLIT_FRACTIONS["test"])
    n_val = round(len(groups) * SPLIT_FRACTIONS["validation"])
    for i, group in enumerate(groups):
        group_split[group] = "test" if i < n_test else "validation" if i < n_test + n_val else "train"

data["split"] = data["line_group"].map(group_split)

# Leakage check: every line group belongs to exactly one split
assert data.groupby("line_group")["split"].nunique().max() == 1

summary = data.groupby("split").agg(
    clips=("audio_name", "size"),
    agents=("agent", "nunique"),
    hours=("duration_s", lambda d: round(d.sum() / 3600, 2)),
    english_only=("english_only", "sum"),
).loc[["train", "validation", "test"]]
summary

,clips,agents,hours,english_only
split,,,,
train,9077,26,4.02,8611
validation,1108,26,0.51,1059
test,1175,26,0.52,1070


In [7]:
# Save the split manifest (used by every later notebook) and the cleaning record
manifest_cols = ["agent", "audio_name", "split", "duration_s", "transcript", "spoken_text", "has_speech", "english_only"]
data[manifest_cols].sort_values(["agent", "audio_name"]).to_csv(f"{RESULTS_DIR}/splits.csv", index=False)

with open(f"{RESULTS_DIR}/data_cleaning.json", "w") as f:
    json.dump({
        "dataset": DATASET,
        "revision": REVISION,
        "agents": AGENTS,
        "min_duration_s": MIN_DURATION_S,
        "cleaning_steps": cleaning_log,
        "split_sizes": summary.reset_index().to_dict(orient="records"),
        "clips_per_agent_per_split": data.groupby(["agent", "split"]).size().unstack().to_dict(orient="index"),
    }, f, indent=2)

print(os.listdir(RESULTS_DIR))

['data_cleaning.json', 'splits.csv']


# Step 3: Choose the 3 candidate pre-trained models

| Key | Hugging Face model | Size | Why it's a candidate |
|---|---|---|---|
| `distilhubert` | `ntu-spml/distilhubert` | ~24M | Compressed (distilled) HuBERT: the smallest and fastest, best suited to the CPU-only Streamlit Cloud |
| `wav2vec2` | `facebook/wav2vec2-base` | ~95M | The standard self-supervised speech model; the most widely used baseline |
| `wavlm` | `microsoft/wavlm-base-plus` | ~95M | Pre-trained with overlapping speakers added, which makes it strong at *speaker* tasks such as identifying who is speaking |

**Why these three:** all are self-supervised speech encoders that take the **same input** (raw 16 kHz audio), so the comparison is fair and the code is shared. They cover a **size/speed trade-off** (24M vs 95M parameters) and include one model built for speaker identity.
*Not chosen:* the Audio Spectrogram Transformer (expects 10 s inputs, so ~1 s clips would be mostly padding, and it is heavier) and Whisper encoders (larger, designed for transcription rather than speaker identity).

**Audio preparation**
- **Resample 48 kHz → 16 kHz:** all three models were pre-trained on 16 kHz speech; other rates degrade their features.
- **Keep at most the first 4 s:** 91.8% of clips are ≤ 4 s and are kept whole. The voice is identifiable within the first seconds, and compute grows with length, so longer clips are trimmed rather than padded.
- **Mono, as stored:** every clip is already single-channel.

In [8]:
import librosa
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

CANDIDATES = {
    "distilhubert": "ntu-spml/distilhubert",
    "wav2vec2": "facebook/wav2vec2-base",
    "wavlm": "microsoft/wavlm-base-plus",
}

TARGET_SR = 16_000     # sampling rate the models were pre-trained on
MAX_SECONDS = 4.0      # longer clips keep their first 4 s

label2id = {agent: i for i, agent in enumerate(AGENTS)}
id2label = {i: agent for agent, i in label2id.items()}


def load_waveform(wav_bytes):
    """WAV bytes -> mono float32 waveform at 16 kHz, at most MAX_SECONDS long."""
    waveform, sr = sf.read(io.BytesIO(wav_bytes), dtype="float32")
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    waveform = waveform[: int(MAX_SECONDS * sr)]
    return librosa.resample(waveform, orig_sr=sr, target_sr=TARGET_SR)


# Decode every clip once (about 1–2 minutes), then free the raw bytes
start = time.perf_counter()
data["waveform"] = data["audio_bytes"].map(load_waveform)
data = data.drop(columns="audio_bytes")
print(f"Decoded {len(data)} clips in {time.perf_counter() - start:.0f}s")


def load_candidate(model_id):
    """Pre-trained speech encoder + a new 26-agent classification head."""
    set_seed(SEED)   # the new head is randomly initialised; seeding makes it reproducible
    feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
    model = AutoModelForAudioClassification.from_pretrained(
        model_id, num_labels=len(AGENTS), label2id=label2id, id2label=id2label,
    )
    # The convolutional feature encoder (low-level acoustic features) is already general;
    # freezing it is the standard recipe and reduces compute and overfitting.
    model.freeze_feature_encoder()
    return feature_extractor, model


for key, model_id in CANDIDATES.items():
    _, m = load_candidate(model_id)
    print(f"{key:<13} {model_id:<28} {m.num_parameters() / 1e6:6.1f}M parameters")
    del m

Decoded 11360 clips in 35s


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/94.0M [00:00<?, ?B/s]

Some weights of HubertForSequenceClassification were not initialized from the model checkpoint at ntu-spml/distilhubert and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of WavLMForSequenceClassification were not initialized from the model checkpoint at microsoft/wavlm-base-plus and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


wavlm         microsoft/wavlm-base-plus      94.6M parameters


# Step 4: Fine-tune and evaluate each model

All three models get **identical** settings:

| Setting | Value | Why |
|---|---|---|
| Epochs | 5 | Enough for a 9k-clip dataset; the best epoch is kept, so extra epochs cannot hurt the final model |
| Learning rate | 3e-5 (10% warm-up) | Standard value for fine-tuning these speech encoders |
| Batch size | 16 | Fits the T4 GPU for all three models |
| Mixed precision (fp16) | On | About 2× faster on the T4, with no meaningful accuracy cost |
| Evaluation | one clip at a time | Matches how the app classifies a clip; batching would pad short clips, which slightly changes the predictions of wav2vec2-base |
| Best epoch | by **validation accuracy** | Accuracy is the course's main metric, and the 26 classes are near-balanced (296–415 training clips each) |

**Metrics:** accuracy (share of clips assigned to the right agent) and macro-F1 (F1 averaged over agents), on validation (for selection) and on test (final score).

In [9]:
import shutil

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import Trainer, TrainingArguments

NUM_EPOCHS = 5
LEARNING_RATE = 3e-5
BATCH_SIZE = 16


class VoiceLines(torch.utils.data.Dataset):
    """One item = {'audio': 16 kHz waveform, 'label': agent id}."""
    def __init__(self, df):
        self.audio = df["waveform"].tolist()
        self.labels = df["agent"].map(label2id).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"audio": self.audio[i], "label": self.labels[i]}


class AudioCollator:
    """Pads a batch of waveforms to the longest one and converts them to model inputs."""
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, batch):
        inputs = self.feature_extractor(
            [item["audio"] for item in batch], sampling_rate=TARGET_SR, padding=True, return_tensors="pt",
        )
        inputs["labels"] = torch.tensor([item["label"] for item in batch])
        return inputs


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }


splits = {name: data[data["split"] == name].reset_index(drop=True) for name in ["train", "validation", "test"]}
datasets = {name: VoiceLines(df) for name, df in splits.items()}
print({name: len(ds) for name, ds in datasets.items()})

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

{'train': 9077, 'validation': 1108, 'test': 1175}


In [10]:
results = {}            # metrics per model (becomes the comparison table)
test_predictions = splits["test"][["agent", "audio_name"]].copy()

for key, model_id in CANDIDATES.items():
    print(f"\n{'=' * 20} {key}: {model_id} {'=' * 20}")
    feature_extractor, model = load_candidate(model_id)

    args = TrainingArguments(
        output_dir=f"checkpoints/{key}",
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,       # one clip at a time, exactly like the app (no padding effects)
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",              # score validation after each epoch...
        save_strategy="epoch",
        load_best_model_at_end=True,        # ...and keep the best epoch
        metric_for_best_model="accuracy",
        save_total_limit=1,
        logging_strategy="epoch",
        remove_unused_columns=False,        # our items are {'audio', 'label'}, prepared by the collator
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=datasets["train"],
        eval_dataset=datasets["validation"],
        data_collator=AudioCollator(feature_extractor),
        processing_class=feature_extractor,   # saved with the model, so pipeline() can load it later
        compute_metrics=compute_metrics,
    )

    train_output = trainer.train()
    val_metrics = trainer.evaluate()                       # validation: used for model selection
    test_output = trainer.predict(datasets["test"])        # test: final score
    test_pred = [id2label[i] for i in np.argmax(test_output.predictions, axis=-1)]
    test_predictions[f"pred_{key}"] = test_pred

    best_epoch = max(
        (log for log in trainer.state.log_history if "eval_accuracy" in log),
        key=lambda log: log["eval_accuracy"],
    )["epoch"]

    results[key] = {
        "model_id": model_id,
        "params_millions": round(model.num_parameters() / 1e6, 1),
        "best_epoch": best_epoch,
        "val_accuracy": val_metrics["eval_accuracy"],
        "val_f1_macro": val_metrics["eval_f1_macro"],
        "test_accuracy": test_output.metrics["test_accuracy"],
        "test_f1_macro": test_output.metrics["test_f1_macro"],
        "train_runtime_s": train_output.metrics["train_runtime"],
        "val_eval_runtime_s": val_metrics["eval_runtime"],
    }

    # Detailed per-model record: per-agent scores, confusion matrix, training curve
    with open(f"{RESULTS_DIR}/{key}_details.json", "w") as f:
        json.dump({
            "model_id": model_id,
            "test_classification_report": classification_report(
                splits["test"]["agent"], test_pred, labels=AGENTS, output_dict=True, zero_division=0),
            "test_confusion_matrix": {
                "labels": AGENTS,   # rows = true agent, columns = predicted agent
                "matrix": confusion_matrix(splits["test"]["agent"], test_pred, labels=AGENTS).tolist(),
            },
            "log_history": trainer.state.log_history,
        }, f, indent=2)

    print(f"test accuracy {results[key]['test_accuracy']:.3f} | test macro-F1 {results[key]['test_f1_macro']:.3f}")

    # Keep the best model for the next steps; drop the checkpoints and free GPU memory
    trainer.save_model(f"models/{key}")
    shutil.rmtree(f"checkpoints/{key}", ignore_errors=True)
    del trainer, model
    torch.cuda.empty_cache()

test_predictions.to_csv(f"{RESULTS_DIR}/test_predictions.csv", index=False)
pd.DataFrame(results).T


==================== distilhubert: ntu-spml/distilhubert ====================


Some weights of HubertForSequenceClassification were not initialized from the model checkpoint at ntu-spml/distilhubert and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.684500,1.988497,0.440433,0.391191
2,1.493400,1.204064,0.720217,0.705388
3,0.853000,0.783505,0.827617,0.825151
4,0.532400,0.643163,0.849278,0.847357
5,0.396500,0.593030,0.859206,0.858644


test accuracy 0.823 | test macro-F1 0.819

==================== wav2vec2: facebook/wav2vec2-base ====================


/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.897100,2.379706,0.233755,0.153104
2,2.046200,1.810664,0.444043,0.375954
3,1.552100,1.465127,0.590253,0.551585
4,1.251900,1.286836,0.617329,0.556940
5,1.092400,1.229705,0.642599,0.589628


/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


test accuracy 0.606 | test macro-F1 0.564


,model_id,params_millions,best_epoch,val_accuracy,val_f1_macro,test_accuracy,test_f1_macro,train_runtime_s,val_eval_runtime_s
distilhubert,ntu-spml/distilhubert,23.7,5.0,0.859206,0.858644,0.822979,0.819366,332.942,7.3509
wav2vec2,facebook/wav2vec2-base,94.6,4.0,0.91787,0.917633,0.904681,0.904942,1067.8404,17.7475
wavlm,microsoft/wavlm-base-plus,94.6,5.0,0.642599,0.589628,0.605957,0.564373,1102.3473,28.9177


# Step 5: Measure CPU runtime through a `pipeline`

The deployed app runs on **Streamlit Cloud, which has no GPU**, so runtime is measured on CPU with the same `pipeline("audio-classification")` call the app will use.

Following the course method, runtime is timed both:
- **with model loading**: `cpu_load_s`, the one-off cost when the app starts
- **without model loading**: `cpu_ms_per_clip`, the cost of each prediction

As a consistency check, the pipeline's predictions must match the Trainer's test predictions from Step 4. This proves the saved model is the one that was evaluated.
(Small differences are possible because training used fp16 on the GPU and the pipeline uses fp32 on the CPU.)

In [11]:
from transformers import pipeline

speed_clips = splits["test"].head(200)

for key in CANDIDATES:
    # With model loading
    start = time.perf_counter()
    classifier = pipeline("audio-classification", model=f"models/{key}", device=-1)   # device=-1 means CPU
    load_s = time.perf_counter() - start

    classifier({"raw": speed_clips["waveform"].iloc[0], "sampling_rate": TARGET_SR})   # warm-up

    # Without model loading: one clip at a time, like the app
    start = time.perf_counter()
    outputs = [classifier({"raw": w, "sampling_rate": TARGET_SR}, top_k=1)[0] for w in speed_clips["waveform"]]
    ms_per_clip = (time.perf_counter() - start) / len(speed_clips) * 1000

    # Consistency check against Step 4
    agreement = np.mean([o["label"] == p for o, p in zip(outputs, test_predictions[f"pred_{key}"].head(200))])

    results[key].update({
        "cpu_load_s": round(load_s, 2),
        "cpu_ms_per_clip": round(ms_per_clip, 1),
        "pipeline_agreement": agreement,
    })
    print(f"{key:<13} load={load_s:5.2f}s  per clip={ms_per_clip:6.1f}ms  agreement with Step 4={agreement:.1%}")
    del classifier

Device set to use cpu


distilhubert  load= 0.09s  per clip= 162.7ms  agreement with Step 4=99.5%


Device set to use cpu


wav2vec2      load= 0.32s  per clip= 477.7ms  agreement with Step 4=100.0%


Device set to use cpu
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


wavlm         load= 0.65s  per clip= 603.6ms  agreement with Step 4=99.5%


# Step 6: Compare the models and pick the best

**Selection rule** (decided before training):
1. Highest **validation accuracy**.
2. If several models are within 0.01 of the best, choose the **fastest on CPU**, because the app runs on CPU.

The test scores are reported for every model but are **not** used to choose. They stay an unbiased estimate of real-world performance.

In [12]:
comparison = pd.DataFrame(results).T.infer_objects().rename_axis("model").reset_index()

best_acc = comparison["val_accuracy"].max()
shortlist = comparison[comparison["val_accuracy"] >= best_acc - 0.01]
BEST = shortlist.sort_values("cpu_ms_per_clip").iloc[0]["model"]
comparison["selected"] = comparison["model"] == BEST

comparison.to_csv(f"{RESULTS_DIR}/model_comparison.csv", index=False)
print(f"Selected model: {BEST} ({CANDIDATES[BEST]})")
comparison

Selected model: wav2vec2 (facebook/wav2vec2-base)


,model,model_id,params_millions,best_epoch,val_accuracy,val_f1_macro,test_accuracy,test_f1_macro,train_runtime_s,val_eval_runtime_s,cpu_load_s,cpu_ms_per_clip,pipeline_agreement,selected
0,distilhubert,ntu-spml/distilhubert,23.7,5.0,0.859206,0.858644,0.822979,0.819366,332.9420,7.3509,0.09,162.7,0.995,False
1,wav2vec2,facebook/wav2vec2-base,94.6,4.0,0.917870,0.917633,0.904681,0.904942,1067.8404,17.7475,0.32,477.7,1.000,True
2,wavlm,microsoft/wavlm-base-plus,94.6,5.0,0.642599,0.589628,0.605957,0.564373,1102.3473,28.9177,0.65,603.6,0.995,False


In [13]:
# Where does the selected model go wrong? The most frequent agent confusions on the test set
details = json.load(open(f"{RESULTS_DIR}/{BEST}_details.json"))
matrix = np.array(details["test_confusion_matrix"]["matrix"])
confusions = [
    (AGENTS[i], AGENTS[j], int(matrix[i, j]))
    for i in range(len(AGENTS)) for j in range(len(AGENTS)) if i != j and matrix[i, j] > 0
]
confusions = pd.DataFrame(confusions, columns=["true agent", "predicted as", "clips"]).sort_values("clips", ascending=False)
confusions.to_csv(f"{RESULTS_DIR}/{BEST}_top_confusions.csv", index=False)

per_agent = pd.DataFrame(details["test_classification_report"]).T.loc[AGENTS, ["precision", "recall", "f1-score", "support"]]
print("Weakest agents (test F1):")
print(per_agent.sort_values("f1-score").head(5).round(3).to_string())
confusions.head(10)

Weakest agents (test F1):
          precision  recall  f1-score  support
skye          0.652   0.878     0.748     49.0
killjoy       1.000   0.644     0.784     45.0
deadlock      0.946   0.714     0.814     49.0
clove         0.812   0.867     0.839     45.0
yoru          1.000   0.741     0.851     54.0


,true agent,predicted as,clips
28,killjoy,skye,11
46,sova,brimstone,5
44,skye,clove,5
43,sage,skye,5
52,yoru,breach,5
31,neon,jett,4
56,yoru,iso,3
23,iso,brimstone,3
9,clove,skye,3
15,deadlock,jett,3


In [14]:
# Record exactly how these results were produced
with open(f"{RESULTS_DIR}/run_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "dataset": DATASET,
        "dataset_revision": REVISION,
        "candidates": CANDIDATES,
        "audio": {"sampling_rate": TARGET_SR, "max_seconds": MAX_SECONDS, "min_seconds": MIN_DURATION_S},
        "hyperparameters": {
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "warmup_ratio": 0.1,
            "batch_size": BATCH_SIZE,
            "fp16": torch.cuda.is_available(),
            "feature_encoder": "frozen",
            "eval_batch_size": 1,
        },
        "selection_rule": "highest val_accuracy; ties within 0.01 go to lowest cpu_ms_per_clip",
        "selected_model": BEST,
        "versions": {"transformers": transformers.__version__, "torch": torch.__version__},
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "run_at_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    }, f, indent=2)

# Step 7: Upload the best model to Hugging Face

The app will load the model from the Hugging Face Hub, together with a model card describing it.

**Token:** add your Hugging Face *write* token as a Colab secret named `HF_TOKEN` (🔑 icon in the left sidebar, then enable notebook access).
Without a token the upload is **skipped** (not an error), so *Restart & Run All* still works for anyone.

In [20]:
from huggingface_hub import ModelCard, get_token

HF_REPO = "eibautistat/valorant-agent-voice-classifier"
best = comparison.set_index("model").loc[BEST]

card = f"""---
library_name: transformers
pipeline_tag: audio-classification
base_model: {CANDIDATES[BEST]}
datasets:
- {DATASET}
language:
- en
tags:
- valorant
- speaker-identification
- audio-classification
metrics:
- accuracy
- f1
---

# VALORANT Agent Voice Classifier

Identifies which of the **{len(AGENTS)} VALORANT agents** is speaking in a voice-line recording.
It is part of a voice-line QA & localization assistant for Riot Games (HKUST ISOM5240 project):
https://github.com/EthanIsaac/valorant-voiceline-qa

- **Fine-tuned from:** [`{CANDIDATES[BEST]}`](https://huggingface.co/{CANDIDATES[BEST]}), selected from 3 candidates on validation accuracy
- **Data:** [`{DATASET}`](https://huggingface.co/datasets/{DATASET}) (revision `{REVISION[:7]}`), split 80/10/10 per agent, grouped by line
- **Input:** mono audio at {TARGET_SR} Hz; the first {MAX_SECONDS:.0f} s are used

## Results (test set, {len(splits["test"])} clips)

| Metric | Value |
|---|---|
| Accuracy | {best["test_accuracy"]:.3f} |
| Macro-F1 | {best["test_f1_macro"]:.3f} |
| CPU time per clip | {best["cpu_ms_per_clip"]:.0f} ms |

## Usage

```python
from transformers import pipeline

classifier = pipeline("audio-classification", model="{HF_REPO}")
classifier("voice_line.wav", top_k=3)
```

## Limitations

Trained on studio-quality in-game voice lines; recordings with music, sound effects or other speakers may be classified less accurately.
Agents: {", ".join(AGENTS)}.
"""

if get_token() is None:
    print("No Hugging Face token found, so the upload is skipped. The published model is at:", f"https://huggingface.co/{HF_REPO}")
else:
    best_model = AutoModelForAudioClassification.from_pretrained(f"models/{BEST}")
    best_extractor = AutoFeatureExtractor.from_pretrained(f"models/{BEST}")
    commit = f"Fine-tuned {CANDIDATES[BEST]} on {DATASET}@{REVISION[:7]} (seed {SEED})"
    best_model.push_to_hub(HF_REPO, commit_message=commit)
    best_extractor.push_to_hub(HF_REPO, commit_message=commit)
    ModelCard(card).push_to_hub(HF_REPO, commit_message="Model card")
    print("Uploaded:", f"https://huggingface.co/{HF_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...x145j84/model.safetensors:   0%|          |  551kB /  378MB            

README.md: 0.00B [00:00, ?B/s]

Uploaded: https://huggingface.co/eibautistat/valorant-agent-voice-classifier


In [21]:
# Test the uploaded model exactly as the app will use it: downloaded from the Hub
hub_classifier = pipeline("audio-classification", model=HF_REPO, device=-1)

for _, row in splits["test"].sample(3, random_state=SEED).iterrows():
    top = hub_classifier({"raw": row["waveform"], "sampling_rate": TARGET_SR}, top_k=1)[0]
    print(f"true: {row['agent']:<10} predicted: {top['label']:<10} ({top['score']:.2f})  \"{row['spoken_text']}\"")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Device set to use cpu


true: neon       predicted: neon       (0.45)  "Our spike's down."
true: harbor     predicted: harbor     (0.95)  "When we are at peace, I would meet you again."
true: clove      predicted: clove      (0.89)  "Holding position."


# Save and download the results

Every file this notebook generated is listed with its **SHA-256 fingerprint** in `SHA256SUMS`.
Rerunning the notebook must reproduce the same fingerprints for the data files (split and cleaning record).
Training files may differ slightly between runs: GPU computation is not bit-for-bit deterministic, and they include runtimes.
Then everything is zipped and downloaded as `results_agent_classifier.zip`.

In [22]:
import hashlib
from pathlib import Path

result_files = sorted(p for p in Path(RESULTS_DIR).rglob("*") if p.is_file() and p.name != "SHA256SUMS")
with open(f"{RESULTS_DIR}/SHA256SUMS", "w") as f:
    for p in result_files:
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        f.write(f"{digest}  {p.relative_to(RESULTS_DIR)}\n")
        print(f"{digest[:16]}…  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size:,} bytes)")

7fc9f92e453b6e8b…  data_cleaning.json  (3,229 bytes)
69c1bb8f5a9e4355…  distilhubert_details.json  (15,511 bytes)
e89da65f74d47d7f…  model_comparison.csv  (656 bytes)
77422138649a9922…  run_config.json  (843 bytes)
72b658b97258de7b…  splits.csv  (1,248,424 bytes)
e5d2fca3af4bafa8…  test_predictions.csv  (49,217 bytes)
cfb51277d5a7192a…  wav2vec2_details.json  (15,390 bytes)
a0b7714cb69911bd…  wav2vec2_top_confusions.csv  (886 bytes)
efdf3d3c04e1c03a…  wavlm_details.json  (15,294 bytes)


In [23]:
!rm -f results_agent_classifier.zip && zip -qrX results_agent_classifier.zip results
from google.colab import files
files.download("results_agent_classifier.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>